
# Microsecond dynamics: 2D-FDC on a single FRET molecule, T3-mode TTTR

Can 2D-FLC resolve *fast* dynamics — hundreds of nanoseconds — on a single
molecule? Here the experiment is reshaped the way the µs regime demands: the
macro-time clock is the **laser period** (T3-mode TCSPC, 25 ns at 40 MHz), the
sample is one immobilized FRET molecule's donor channel at 500 kcps, and the
two conformational states differ in FRET efficiency (E = 0.2 / 0.8, i.e.
donor lifetimes 3.2 ns and 0.8 ns at tau0 = 4 ns) with equal exchange rates,
so the relaxation time is 1/(k01+k10).

The coupling statistic D(dT) — total variation between the pair distribution
and the product of its marginals — is fitted log-linearly above the noise
floor, the same estimator as the slower benchmark
(``plot_fdc_2d_dynamics_resolution.py``), for comparability.

Measured on this machine (2026-08-16, three seeds per point, 5 s stream
~2.5 M photons each, lag window ddT = 2 clocks = 75 ns effective span):

=========== ============= ==============
true        fitted (3-seed mean)  error
=========== ============= ==============
200 ns      155 ns        22 %
500 ns      396 ns        21 %
1.0 us      820 ns        18 %
2.0 us      1.67 us       17 %
5.0 us      4.24 us       15 %
10.0 us     8.36 us       16 %
=========== ============= ==============

So yes: everything from **200 ns to 10 us is recovered within 25%**, with a
tight seed spread (+/-3%) and a consistent -15…-22% bias that belongs to the
simple estimator (a weighted exponential fit with baseline would shave it;
kept identical to the slow benchmark instead). The resolution floor is again
the *window*: 200 ns sits 2.7x above the 75 ns effective lag-window span, and
pushing below that needs a narrower window or a faster laser, not a different
photon pass.

Two honest caveats. The simulator emits Poisson — no detector afterpulsing,
no dead-time pile-up, no dark counts — and real µs-lag correlations fight
exactly those instrumental artifacts; treat these numbers as the
physics-limited ceiling, not an end-to-end instrument forecast. And the
molecule here is immobilized (one continuous stream); a burst experiment on
freely diffusing molecules aggregates its bursts, which the pair pass takes
in stride (it is one stream walk) but which adds burst-selection statistics
this example does not model.

Method and provenance as in ``plot_fdc_2d.py``: Ishii & Tahara, J. Phys.
Chem. B 117(39), 11414-11422 and 11423-11432 (2013), doi:10.1021/jp406861u,
doi:10.1021/jp406864e; Kondo et al., Proc. Natl. Acad. Sci. USA 116(23),
11247-11252 (2019), doi:10.1073/pnas.1821207116 — the µs-ms single-molecule
application whose acquisition geometry this example imitates.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import tttrlib

N_MICRO = 256
LASER_NS = 25.0           # laser period: the macro clock (T3), in ns
DT_S = LASER_NS * 1e-9    # one window per laser period, in seconds
MICRO_DT = LASER_NS / N_MICRO
L = 12                    # log bins per matrix axis
DDT = 2                   # lag window: +/-1 clock, 75 ns effective span
TAU0 = 4.0                # donor-only lifetime, ns
E1, E2 = 0.2, 0.8         # the two FRET states
TAUS = (TAU0 * (1 - E1), TAU0 * (1 - E2))     # 3.2 / 0.8 ns
RATE_CPS = 500_000.0      # one immobilized molecule, donor channel
T_STREAM = 5.0            # seconds
SEEDS = (11, 23, 31)
RELAX = np.array([0.2e-6, 0.5e-6, 1e-6, 2e-6, 5e-6, 10e-6])


def _vd(x):
    return tttrlib.VectorDouble([float(v) for v in x])


def simulate(k, seed):
    """One immobile FRET molecule: tau set by each state's efficiency,
    equal exchange rates k both ways (relaxation = 1/(2k))."""
    system = tttrlib.SimSystem()
    for tau in TAUS:
        sp = tttrlib.SimSpecies()
        sp.D = 0.0
        sp.q = _vd([RATE_CPS])
        sp.decay = tttrlib.SimDecay.multi_exponential(
            _vd([1.0]), _vd([tau]), N_MICRO, MICRO_DT)
        system.add_species(sp)
    system.set_rate_matrices(_vd([0.0] * 4), _vd([0.0, k, k, 0.0]))
    system.set_background(_vd([0.0]))
    system.set_box(50.0, 50.0)
    system.add_fluorophore(0.0, 0.0, 0.0, 0, False)

    it = tttrlib.SimIntegrator()
    it.dt = DT_S                       # seconds; micro-time fields are ns
    it.n_channels = 1
    it.n_ph_max = 10 ** 12
    it.max_windows = int(T_STREAM / DT_S)
    it.n_microtime_channels = N_MICRO
    it.microtime_resolution = MICRO_DT
    it.laser_period = LASER_NS
    it.seed_diffusion = seed
    it.seed_emission = seed + 1

    eng = tttrlib.SimEngine(
        system, tttrlib.SimGrid.gaussian3d(0.3, 2.0, 4.0, 8.0, 0.2, 1.0),
        tttrlib.VectorSimGrid([]), it)
    eng.run()
    return (np.asarray(eng.macro_window(), dtype=np.int64),
            np.asarray(eng.micro_time(), dtype=np.int64))


def d_curve(macro, micro, lags):
    """Coupling D at each lag (total variation vs the product of marginals)."""
    lags = np.asarray(lags, dtype=np.int64)
    out = np.zeros(lags.size * L * L, dtype=np.int64)
    tttrlib.fdc_scan_log(macro, micro, lags, DDT, 0, N_MICRO - 1, L, 4, out)
    d = []
    for m in out.reshape(lags.size, L, L):
        p = m.astype(float) / m.sum()
        d.append(0.5 * np.abs(p - np.outer(p.sum(1), p.sum(0))).sum())
    return np.asarray(d)


def lag_grid(relax_s):
    """Lags straddling the relaxation, every one clear of the window span
    (lag > DDT/2 = 1 clock; self-pairs would otherwise spike the diagonal)."""
    lo = max(4, int(round(0.15 * relax_s / DT_S)))
    hi = max(48, int(round(6.0 * relax_s / DT_S)))
    return np.unique(np.round(np.geomspace(lo, hi, 8)).astype(np.int64))


def fit_relaxation(lags_s, d):
    y = d - d[-1]
    usable = y > 0.25 * y[0]
    if usable.sum() < 3 or y[0] <= 0:
        return np.nan
    slope = np.polyfit(lags_s[usable], np.log(y[usable]), 1)[0]
    return -1.0 / slope


fitted = np.full((RELAX.size, len(SEEDS)), np.nan)
for i, relax in enumerate(RELAX):
    lags = lag_grid(relax)
    for j, seed in enumerate(SEEDS):
        macro, micro = simulate(0.5 / relax, seed)
        d = d_curve(macro, micro, lags)
        fitted[i, j] = fit_relaxation(lags * DT_S, d)
    print(f"true {relax * 1e9:6.0f} ns -> fitted "
          f"{np.nanmean(fitted[i]) * 1e9:8.1f} ns", flush=True)

mean = np.nanmean(fitted, axis=1)
err = 100.0 * np.abs(mean - RELAX) / RELAX

# one D curve for the figure: the hardest case, 200 ns
lags = lag_grid(RELAX[0])
macro, micro = simulate(0.5 / RELAX[0], SEEDS[0])
d_slow = d_curve(macro, micro, lags)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
ax.loglog(RELAX * 1e9, RELAX * 1e9, "k--", lw=1, label="identity")
for j in range(len(SEEDS)):
    ax.loglog(RELAX * 1e9, fitted[:, j] * 1e9, "o", ms=4, alpha=0.4,
              color="tab:blue")
ax.loglog(RELAX * 1e9, mean * 1e9, "o-", color="tab:blue",
          label="fitted (mean of 3 seeds)")
ax.axvspan(25, 75, color="tab:red", alpha=0.15,
           label="below window span (75 ns)")
ax.set_xlabel("true relaxation (ns)")
ax.set_ylabel("fitted relaxation (ns)")
ax.set_title("single FRET molecule, T3 mode\n"
             f"errors {err.min():.0f}-{err.max():.0f} % (all < 25 %)")
ax.legend(fontsize=8)

ax = axes[1]
ax.semilogx(lags * DT_S * 1e9, d_slow, "o-", color="tab:blue",
            label="coupling D(dT), 200 ns case")
span = d_slow[0] - d_slow[-1]
tt = np.geomspace(lags[0], lags[-1], 100) * DT_S * 1e9
ax.semilogx(tt, span * np.exp(-tt / mean[0]) + d_slow[-1], "-",
            color="tab:red", lw=1.5,
            label=f"fit {mean[0] * 1e9:.0f} ns (true 200 ns)")
ax.set_xlabel("lag dT (ns)")
ax.set_ylabel("D (total variation)")
ax.set_title("the hardest case: 200 ns relaxation")
ax.legend(fontsize=8)

fig.suptitle("2D-FDC microsecond dynamics: 40 MHz clock, 500 kcps, "
             f"E = {E1}/{E2} (tau = {TAUS[0]}/{TAUS[1]} ns)", y=1.03)
fig.tight_layout()
plt.show()